# Hybrid Full-xTB Multi-Run Training

This notebook launches 5 isolated `hybrid_full_xtb` training runs with fixed settings and different seeds.

It is designed for pattern-finding, not single-run tuning:
- `30` epochs max per run
- early stopping on `site_top1`
- fresh subprocess per seed
- per-seed output and artifact directories
- per-seed plain-text console log and structured `.jsonl`


In [ ]:
%cd /content/enzyme_Software
!git pull origin main


In [ ]:
from datetime import datetime
from pathlib import Path

RUN_TAG = datetime.now().strftime("multirun_%Y%m%d_%H%M%S")
SEEDS = [11, 22, 33, 44, 55]
EPOCHS = 30
BATCH_SIZE = 16
LR = "3e-5"
WD = "1e-4"
BACKBONE_FREEZE_EPOCHS = 5
FREEZE_NEXUS_MEMORY = 0
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_METRIC = "site_top1"

DATASET = "data/prepared_training/main8_site_conservative_singlecyp_clean_symm.json"
STRUCTURE_SDF = "3D structures.sdf"
XTB_CACHE_DIR = "/content/drive/MyDrive/enzyme_hybrid_lnn/cache/full_xtb"
MANUAL_CACHE_DIR = "/content/drive/MyDrive/enzyme_hybrid_lnn/cache/manual_engine_full"
OUTPUT_ROOT = "/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/hybrid_full_xtb_multirun"
ARTIFACT_ROOT = "/content/drive/MyDrive/enzyme_hybrid_lnn/artifacts/hybrid_full_xtb_multirun"

print("RUN_TAG=", RUN_TAG)
print("SEEDS=", SEEDS)
print("OUTPUT_ROOT=", OUTPUT_ROOT)
print("ARTIFACT_ROOT=", ARTIFACT_ROOT)


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/enzyme_Software")


def _base_env():
    env = os.environ.copy()
    for key in list(env):
        if key.startswith("HYBRID_COLAB_"):
            env.pop(key, None)
    env["HYBRID_COLAB_PRESET"] = "balanced"
    env["HYBRID_COLAB_LOCK_PRESET_POLICY"] = "1"
    env["HYBRID_COLAB_DATASET"] = DATASET
    env["HYBRID_COLAB_STRUCTURE_SDF"] = STRUCTURE_SDF
    env["HYBRID_COLAB_XTB_CACHE_DIR"] = XTB_CACHE_DIR
    env["HYBRID_COLAB_MANUAL_CACHE_DIR"] = MANUAL_CACHE_DIR
    env["HYBRID_COLAB_EPOCHS"] = str(EPOCHS)
    env["HYBRID_COLAB_BATCH_SIZE"] = str(BATCH_SIZE)
    env["HYBRID_COLAB_LR"] = LR
    env["HYBRID_COLAB_WD"] = WD
    env["HYBRID_COLAB_SPLIT_MODE"] = "scaffold_source_size"
    env["HYBRID_COLAB_SITE_LABELED_ONLY"] = "1"
    env["HYBRID_COLAB_COMPUTE_XTB_IF_MISSING"] = "0"
    env["HYBRID_COLAB_FREEZE_NEXUS_MEMORY"] = str(FREEZE_NEXUS_MEMORY)
    env["HYBRID_COLAB_BACKBONE_FREEZE_EPOCHS"] = str(BACKBONE_FREEZE_EPOCHS)
    env["HYBRID_COLAB_EARLY_STOPPING_PATIENCE"] = str(EARLY_STOPPING_PATIENCE)
    env["HYBRID_COLAB_EARLY_STOPPING_METRIC"] = EARLY_STOPPING_METRIC
    env["HYBRID_COLAB_DISABLE_PRECEDENT_LOGBOOK"] = "1"
    env["HYBRID_COLAB_INCLUDE_XENOSITE"] = "1"
    env["HYBRID_COLAB_XENOSITE_TOPK"] = "1"
    env["HYBRID_COLAB_LIVE_WAVE_VOTE_INPUTS"] = "1"
    env["HYBRID_COLAB_LIVE_ANALOGICAL_VOTE_INPUTS"] = "1"
    return env


def run_one(seed: int):
    env = _base_env()
    seed_name = f"seed_{seed}"
    output_dir = Path(OUTPUT_ROOT) / RUN_TAG / seed_name
    artifact_dir = Path(ARTIFACT_ROOT) / RUN_TAG / seed_name
    output_dir.mkdir(parents=True, exist_ok=True)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    env["HYBRID_COLAB_SEED"] = str(seed)
    env["HYBRID_COLAB_OUTPUT_DIR"] = str(output_dir)
    env["HYBRID_COLAB_ARTIFACT_DIR"] = str(artifact_dir)
    env["HYBRID_COLAB_CONSOLE_LOG"] = str(artifact_dir / f"hybrid_full_xtb_console_seed_{seed}.log")
    env["HYBRID_COLAB_CONSOLE_JSONL"] = str(artifact_dir / f"hybrid_full_xtb_console_seed_{seed}.jsonl")

    print(f"\\n=== Launching {seed_name} ===", flush=True)
    subprocess.check_call(
        [sys.executable, "scripts/colab_train_hybrid_lnn.py"],
        cwd=str(REPO_DIR),
        env=env,
    )
    return {
        "seed": seed,
        "output_dir": str(output_dir),
        "artifact_dir": str(artifact_dir),
        "console_log": env["HYBRID_COLAB_CONSOLE_LOG"],
        "console_jsonl": env["HYBRID_COLAB_CONSOLE_JSONL"],
    }


def latest_report(artifact_dir: str):
    reports = sorted(Path(artifact_dir).glob("hybrid_full_xtb_report_*.json"))
    return reports[-1] if reports else None


def report_summary(seed: int, artifact_dir: str):
    report_path = latest_report(artifact_dir)
    row = {
        "seed": seed,
        "report_path": str(report_path) if report_path else "",
        "best_epoch": None,
        "history_len": None,
        "best_val_top1": None,
        "test_top1": None,
        "test_top3": None,
        "test_auc": None,
        "wave_reliability_mean": None,
        "analogical_gate_mean": None,
    }
    if report_path is None:
        return row
    report = json.loads(report_path.read_text())
    row["best_epoch"] = report.get("best_epoch")
    row["history_len"] = report.get("history_len")
    row["best_val_top1"] = report.get("best_val_top1")
    row["test_top1"] = report.get("site_top1")
    row["test_top3"] = report.get("site_top3")
    row["test_auc"] = report.get("site_auc")
    row["wave_reliability_mean"] = report.get("nexus_wave_reliability_mean")
    row["analogical_gate_mean"] = report.get("nexus_analogical_gate_mean")
    return row


In [ ]:
run_records = []
for seed in SEEDS:
    run_records.append(run_one(seed))

print("\\nCompleted seeds:")
for record in run_records:
    print(record)


In [ ]:
import pandas as pd

summary_rows = [report_summary(record["seed"], record["artifact_dir"]) for record in run_records]
summary_df = pd.DataFrame(summary_rows).sort_values("seed").reset_index(drop=True)
summary_df


In [ ]:
summary_path = Path(ARTIFACT_ROOT) / RUN_TAG / "multirun_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print(summary_path)
